# Формулировки промпта для T-lite: формат и поиск

В пробном прогоне (`tlite_episode_pilot.ipynb`, `runs/tlite_pilot/20261019_2228/`) исходная T-lite не начинала первый
слайд строкой `#SLIDE` (в двух ответах из четырёх `#SLIDE` не было вовсе), писала `#CHART` без `type` и слова вместо
чисел в `y`, LaTeX внутри `#TEXT`, комментарии после значений и ни разу не вызвала поиск. Здесь проверяю, что из этого
исправляется промптом ещё до SFT. `prompts.py` не меняю, пока не выбран лучший вариант.

Варианты системного промпта:
- **A** — текущий `system_prompt`;
- **B** — A + правило «каждый слайд, включая первый, начинается `#SLIDE`», список частых ошибок формата (`type=` у `#CHART`,
  в `y` только числа, LaTeX только в `#FORMULA`, без комментариев в строке) и пример из двух слайдов (фотоэффект,
  как в промпте составителей SFT);
- **C** — короткий: только теги, ограничения формата, те же правило, ошибки и пример, без разделов «Как выполнять запрос»
  и «Слайды»;
- **D** — B + прямое указание в разделе поиска: прежде чем писать слайды, найти в Википедии определения, даты и числа.

Часть 1 — 8 заданий validation × 4 варианта, поиск и картинки есть (32 эпизода). Часть 2 — 4 запроса, где без точных
фактов не обойтись, варианты B и D (8 эпизодов). Температура 0.9, как будет в RL. Seed фиксирован по заданию — прогон
воспроизводим; если разница между вариантами мала, повторить два лучших с другим `SEED`. В примере есть `#IMAGE`, поэтому
все запросы здесь с картинками.

Что считаю по каждому эпизоду: слайдов разобрано и сколько просили, начинается ли ответ с `#SLIDE` (после `#PLAN`), ошибки
разбора по кодам, вызовы инструментов (сколько, какие, длина запроса в словах, ошибочные), принятые и отклонённые `#SOURCE`,
токены и время.

Результаты — `runs/prompt_experiments/<время>/`: тексты промптов `prompts.json`; по каждому эпизоду
`<часть>_<вариант>_<id>.txt` (эпизод целиком), `_answer.txt` (ответ для парсера), `.pptx`; таблица `summary.csv`
(дописывается после каждого эпизода) и итоги по вариантам `totals.csv`.

In [ ]:
import csv, dataclasses, json, os, statistics, sys, time
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "4"  # перед запуском проверить nvidia-smi: карта должна быть свободна
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
import torch
from presentation_rl.dsl import parse
from presentation_rl.environment import build_prompt, run_episode
from presentation_rl.model_loader import hf_generator, load_model
from presentation_rl.prompts import BASE, SEARCH, system_prompt
from presentation_rl.rendering import build_presentation, pick_theme
from presentation_rl.tasks import PresentationRequest, load_tasks
from presentation_rl.tools.cache import WikiCache
from presentation_rl.tools.wiki import WikiClient
from presentation_rl.tools.wiki_tools import WikiTools

print(torch.__version__, torch.cuda.get_device_name(0))
CONTACT = os.environ.get("WIKI_CONTACT", "https://github.com/doshik722")
cache = WikiCache(ROOT / "data" / "sources" / "wiki_cache", WikiClient(contact=CONTACT))
OUT = ROOT / "runs" / "prompt_experiments" / time.strftime("%Y%m%d_%H%M")
OUT.mkdir(parents=True, exist_ok=True)
SEED = 0

In [ ]:
# В пилоте поиск не вызывался ни разу, так что доступ к Википедии с сервера ещё не проверен — проверяю до загрузки модели.
print(WikiTools(cache).search("фотоэффект")[:600])

In [ ]:
start = time.time()
model, tokenizer = load_model("t-tech/T-lite-it-2.1", dtype="float16")
generate = hf_generator(model, tokenizer, temperature=0.9)
print(f"загрузка {time.time() - start:.0f} с, память {torch.cuda.memory_allocated() / 2**30:.1f} ГБ")

## Варианты промпта

Варианты собираю из `BASE` и `SEARCH` вставками после строк-якорей: если промпт в `prompts.py` поменяется, якорь
не найдётся и ячейка упадёт, а не соберёт молча другой промпт.

In [ ]:
def insert_after(text, anchor, addition):
    assert text.count(anchor) == 1, f"якорь не найден или не один: {anchor}"
    return text.replace(anchor, anchor + addition)


SLIDE_RULE = ("\n- Каждый слайд, включая первый (титульный), начинается отдельной строкой `#SLIDE`; строки до первого "
              "`#SLIDE`, кроме `#PLAN`, программа отбрасывает.")
MISTAKES = """

Частые ошибки формата — не допускай их:
- у `#CHART` всегда есть `type=`; в `y` — только числа, по одному на каждую категорию из `x`;
- формулы — только строкой `#FORMULA`; в `#TITLE`, `#TEXT` и `#BULLET` не пиши LaTeX и знаки `\\(`, `\\[`, `$`;
- в строке один тег и его значение; комментарии после значения не пиши."""
EXAMPLE = r"""
## Пример

Начало презентации на другую тему — образец формата, а не содержания и не длины:

#PLAN: открытие → уравнение Эйнштейна → красная граница → применение
#SLIDE
#TITLE: Фотоэффект: как свет выбивает электроны
#TEXT: Почему ультрафиолет разряжает цинковую пластину, а яркий красный свет — нет? Ответ на этот вопрос привёл к идее квантов света.
#IMAGE: металлическая пластина под лучом ультрафиолетовой лампы в физической лаборатории
#SLIDE
#TITLE: Красная граница и выбор металла
#TEXT: Если энергии фотона не хватает на работу выхода, электроны не вылетают при любой яркости. Наименьшая частота — красная граница — у каждого металла своя.
#FORMULA: \nu_{min} = \frac{A}{h}
#CHART: type=bar; title=Работа выхода, эВ; x=цезий,натрий,цинк; y=1.9,2.3,4.2
#TEXT: У цинка граница лежит в ультрафиолете, а у цезия — в видимом свете, поэтому цезий используют в фотоэлементах.
#NOTES: Пояснить, что работа выхода зависит и от состояния поверхности, поэтому в справочниках значения немного различаются. Спросить зал, хватит ли зелёного света, около 2,3 эВ, чтобы выбить электрон из цинка.
"""
SEARCH_FIRST = ("- Прежде чем писать слайды, найди в Википедии определения, даты и числа по теме: `search`, затем `open` "
                "или `read` нужных статей. Слайды пиши после этого, по найденному.\n")

BASE_B = insert_after(BASE, "- Перед слайдами — строка `#PLAN`: части презентации по порядку через «→».", SLIDE_RULE)
BASE_B = insert_after(BASE_B, "химию пиши как `6\\text{CO}_2`.", MISTAKES)
SEARCH_D = insert_after(SEARCH, "## Поиск по Википедии («Поиск: есть»)\n\n", SEARCH_FIRST)

# C: вводный абзац, пара правил из «Как выполнять запрос» одной строкой и раздел «Формат» без раздела «Слайды».
INTRO = BASE[:BASE.index("## Как выполнять запрос")]
FORMAT = BASE[BASE.index("## Формат"):BASE.index("## Слайды")].rstrip("\n")
BASE_C = (INTRO + "Язык — русский, если не попросили другой; число слайдов — из строки «Слайдов», если она есть; "
          "первый слайд — титульный; «Картинки: нет» — без `#IMAGE`.\n\n" + FORMAT + SLIDE_RULE + MISTAKES + "\n")


def variant(base, search):
    return lambda request: base + (search if request.search else "") + EXAMPLE


VARIANTS = {"A": system_prompt, "B": variant(BASE_B, SEARCH), "C": variant(BASE_C, SEARCH), "D": variant(BASE_B, SEARCH_D)}

probe = PresentationRequest("Сделай презентацию о фотоэффекте.")  # поиск и картинки есть
prompts = {name: make(probe) for name, make in VARIANTS.items()}
(OUT / "prompts.json").write_text(json.dumps(prompts, ensure_ascii=False, indent=1), encoding="utf-8")
for name, text in prompts.items():
    print(name, len(tokenizer.encode(text, add_special_tokens=False)), "токенов")

In [ ]:
print(prompts["D"])  # самый длинный вариант целиком

## Задания

Часть 1 — 4 технических и 4 гуманитарных задания validation; t367, t369, t366 были в пилоте. Число слайдов задано
в t367 (10) и t381 (7–10). В t366 просят один слайд, в t369 — больше 8, но `slide_range` этого не распознаёт
(в таблице пусто) — эти два смотрю вручную.

Часть 2 — запросы, где нужны точные даты и числа; в двух поиск по Википедии назван прямо, в двух — нет.

In [ ]:
tasks = {t.id: t for t in load_tasks(ROOT / "data" / "tasks" / "tasks.json", split="validation")}
PART1 = ["t367", "t369", "t376", "t381", "t366", "t378", "t386", "t389"]
PART2 = {
    "s1": PresentationRequest("Сделай презентацию о Полтавской битве по материалам Википедии, с указанием источников.",
                              domain="humanities"),
    "s2": PresentationRequest("Презентация о крупнейших ГЭС России: мощность, год ввода в эксплуатацию, река. "
                              "Данные возьми из Википедии.", slides=8, domain="technical"),
    "s3": PresentationRequest("Сделай презентацию о космическом телескопе «Джеймс Уэбб»: когда запущен, какого размера "
                              "зеркало, где находится и что уже открыл.", domain="technical"),
    "s4": PresentationRequest("Презентация о нобелевских лауреатах по литературе из России и СССР: кто, когда и за что "
                              "получил премию.", domain="humanities"),
}
for key in PART1:
    request = tasks[key].request
    print(key, request.slide_range(), "|", request.text[:90])

## Прогон

In [ ]:
FIELDS = ["part", "variant", "id", "finish", "prompt_tokens", "generated_tokens", "seconds", "slides", "requested",
          "in_range", "starts_with_slide", "slide_tags", "parse_rate", "parse_errors", "error_codes", "calls",
          "call_names", "queries", "query_words", "call_errors", "sources_ok", "sources_bad", "render"]
rows = []


def starts_with_slide(answer):
    # Первая содержательная строка ответа (не #PLAN и не ```) — #SLIDE?
    for line in answer.splitlines():
        line = line.strip()
        if line and not line.startswith("```") and not line.upper().startswith("#PLAN"):
            return line.upper().startswith("#SLIDE")
    return False


def run(part, name, key, request, number):
    tools = WikiTools(cache) if request.search else None
    prompt_ids = build_prompt(tokenizer, VARIANTS[name](request), request)
    torch.manual_seed(SEED + number)
    start = time.time()
    episode = run_episode(prompt_ids, tokenizer, generate, tools, max_new_tokens=8000)
    seconds = time.time() - start
    deck = parse(episode.answer)
    stem = f"{part}_{name}_{key}"
    report = build_presentation(deck, OUT / f"{stem}.pptx", pick_theme(request.theme, request.text),
                                sources=tools.source if tools else None, show_sources=request.sources)
    (OUT / f"{stem}.txt").write_text(tokenizer.decode(episode.ids), encoding="utf-8")
    (OUT / f"{stem}_answer.txt").write_text(episode.answer, encoding="utf-8")

    requested = request.slide_range()
    queries = [c["argument"] for c in episode.calls if c["name"] == "search" and not c["error"]]
    sources = [item.status for item in report.items if item.kind == "SOURCE"]
    row = {
        "part": part, "variant": name, "id": key, "finish": episode.finish,
        "prompt_tokens": len(episode.prompt_ids), "generated_tokens": episode.generated_tokens,
        "seconds": round(seconds), "slides": len(deck.slides),
        "requested": "" if requested is None else "–".join(str(n) for n in sorted(set(requested))),
        "in_range": "" if requested is None else requested[0] <= len(deck.slides) <= requested[1],
        "starts_with_slide": starts_with_slide(episode.answer),
        "slide_tags": sum(line.strip().upper() == "#SLIDE" for line in episode.answer.splitlines()),
        "parse_rate": round(deck.parse_rate, 3), "parse_errors": len(deck.errors),
        "error_codes": json.dumps(deck.error_counts(), ensure_ascii=False),
        "calls": len(episode.calls), "call_names": ",".join(c["name"] or "?" for c in episode.calls),
        "queries": " | ".join(queries),
        "query_words": round(statistics.mean(len(q.split()) for q in queries), 1) if queries else "",
        "call_errors": sum(bool(c["error"]) for c in episode.calls),
        "sources_ok": sources.count("ok"), "sources_bad": sources.count("bad_source"),
        "render": json.dumps({k: v for k, v in report.status_counts().items() if k != "ok"}, ensure_ascii=False),
    }
    rows.append(row)
    # utf-8-sig — чтобы Excel на компьютере открыл кириллицу без перекодировки.
    with open(OUT / "summary.csv", "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, FIELDS)
        writer.writeheader()
        writer.writerows(rows)
    print(f"{stem}: {row['finish']}, {row['generated_tokens']} ток., {row['seconds']} с, слайдов {row['slides']}"
          f" (просили {row['requested'] or '—'}), с #SLIDE: {row['starts_with_slide']}, ошибок {row['parse_errors']},"
          f" вызовов {row['calls']}, источников {row['sources_ok']}/{row['sources_ok'] + row['sources_bad']}")
    return row


def totals(part):
    result = []
    for name in VARIANTS:
        group = [r for r in rows if r["part"] == part and r["variant"] == name]
        if not group:
            continue
        ranged = [r for r in group if r["in_range"] != ""]
        result.append({
            "part": part, "variant": name, "episodes": len(group),
            "no_slides": sum(r["slides"] == 0 for r in group),
            "starts_with_slide": sum(r["starts_with_slide"] for r in group),
            "in_range": f"{sum(r['in_range'] is True for r in ranged)}/{len(ranged)}",
            "slides_mean": round(statistics.mean(r["slides"] for r in group), 1),
            "parse_rate_mean": round(statistics.mean(r["parse_rate"] for r in group), 3),
            "parse_errors": sum(r["parse_errors"] for r in group),
            "with_calls": sum(r["calls"] > 0 for r in group),
            "calls": sum(r["calls"] for r in group),
            "call_errors": sum(r["call_errors"] for r in group),
            "sources_ok": sum(r["sources_ok"] for r in group),
            "sources_bad": sum(r["sources_bad"] for r in group),
            "not_stopped": sum(r["finish"] != "stop" for r in group),
            "tokens_mean": round(statistics.mean(r["generated_tokens"] for r in group)),
            "seconds_mean": round(statistics.mean(r["seconds"] for r in group)),
        })
    return result


def save_totals():
    table = totals("p1") + totals("p2")
    with open(OUT / "totals.csv", "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, list(table[0]))
        writer.writeheader()
        writer.writerows(table)
    columns = list(table[0])
    widths = [max(len(c), *(len(str(r[c])) for r in table)) for c in columns]
    print("  ".join(c.ljust(w) for c, w in zip(columns, widths)))
    for r in table:
        print("  ".join(str(r[c]).ljust(w) for c, w in zip(columns, widths)))

### Часть 1: формат (8 заданий × A, B, C, D)

Порядок — по заданиям: если прогон прервётся, у готовых заданий будут все четыре варианта.

In [ ]:
for number, key in enumerate(PART1):
    request = dataclasses.replace(tasks[key].request, search=True, images=True)
    for name in "ABCD":
        run("p1", name, key, request, number)
save_totals()

### Часть 2: вызывает ли T-lite поиск (4 запроса × B, D)

In [ ]:
for number, (key, request) in enumerate(PART2.items(), start=100):
    for name in "BD":
        run("p2", name, key, request, number)
save_totals()

In [ ]:
# Первый эпизод с вызовами инструментов целиком — как его видела модель (без системного промпта).
with_calls = [r for r in rows if r["calls"]]
if with_calls:
    r = with_calls[0]
    text = (OUT / f"{r['part']}_{r['variant']}_{r['id']}.txt").read_text(encoding="utf-8")
    print(text[text.index("<|im_start|>user"):])
else:
    print("ни одного вызова инструментов")